# 07. M1 normal vs fault event 재라벨링 학습

목표는 기존 조기탐지 후보 분류가 아니라, M1 normal window와 M1 fault record window가 센서 통계 feature로 구분되는지 확인하는 것이다.

- positive 정의: M1 `faults.csv`에 존재하는 fault record
- positive window: `Report date - 7 days`부터 `Report date`까지
- negative window: M1 `normal_events.csv`의 7일 window
- feature: M1 공통 10개 센서 x 7개 통계 = 70개
- Event 20: 낮은 coverage로 학습 제외, audit 보존
- Event 69: 요청 기준에 따라 학습 제외, audit 보존
- Event 34: `fault_label_unknown`으로 기록하고 포함/제외 민감도 비교
- Event 67: 장기 anomaly flag로 기록하고 포함/제외 민감도 비교

## 실행 원칙

이번 실험은 fault event 상태 구분 가능성 검증이다. 조기탐지 실험이 아니며, `unknown` fault label은 negative나 비라벨 이벤트로 단정하지 않는다.

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings
import zipfile

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
OUTPUT_DIR = ROOT / "07_데이터산출물"

AUDIT_PATH = OUTPUT_DIR / "m1_fault_event_audit.csv"
FEATURES_PATH = OUTPUT_DIR / "m1_fault_event_features.csv"
DATASET_SUMMARY_PATH = OUTPUT_DIR / "m1_fault_event_dataset_summary.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_fault_event_cv_metrics.csv"
CV_PREDICTIONS_PATH = OUTPUT_DIR / "m1_fault_event_cv_predictions.csv"
THRESHOLD_METRICS_PATH = OUTPUT_DIR / "m1_fault_event_threshold_metrics.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_fault_event_feature_importance.csv"
DECISION_MATRIX_PATH = OUTPUT_DIR / "m1_fault_event_decision_matrix.csv"
REPORT_PATH = OUTPUT_DIR / "07_M1_normal_vs_fault_event_보고서.md"

SOURCE_PREFIX = "manufacturer 1"
RANDOM_STATE = 42
POSITIVE_LABEL = "fault_event"
EVENT20_ID = 20
EVENT34_ID = 34
EVENT67_ID = 67
EVENT69_ID = 69

COMMON_SENSOR_COLUMNS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
FEATURE_COLUMNS = [f"{sensor}__{stat}" for sensor in COMMON_SENSOR_COLUMNS for stat in SENSOR_STATS]
THRESHOLDS = [0.3, 0.4, 0.5, 0.6]
LABEL_TO_BINARY = {"normal": 0, POSITIVE_LABEL: 1}
BINARY_TO_LABEL = {0: "normal", 1: POSITIVE_LABEL}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.exists(), ZIP_PATH

## 1. 원본 metadata 로드

In [2]:
def read_metadata(name: str) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(f"{SOURCE_PREFIX}/{name}") as file:
            return pd.read_csv(file, sep=";")


def parse_datetime_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


faults = parse_datetime_columns(
    read_metadata("faults.csv"),
    ["Report date", "Possible anomaly start", "Possible anomaly end", "Training start", "Training end"],
)
normal_events = parse_datetime_columns(
    read_metadata("normal_events.csv"),
    ["Event start", "Event end", "Training start", "Training end"],
)
disturbances = parse_datetime_columns(read_metadata("disturbances.csv"), ["Event start"])

faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)
normal_events["Event ID"] = normal_events["Event ID"].astype(int)
normal_events["substation ID"] = normal_events["substation ID"].astype(int)
disturbances["substation ID"] = disturbances["substation ID"].astype(int)

print("fault rows:", len(faults))
print("normal rows:", len(normal_events))
print("fault IDs:", sorted(faults["Event ID"].tolist()))

fault rows: 33
normal rows: 35
fault IDs: [1, 3, 5, 6, 7, 10, 11, 13, 15, 20, 23, 24, 29, 32, 34, 36, 37, 38, 40, 44, 45, 47, 49, 52, 53, 57, 60, 62, 63, 64, 65, 67, 69]


## 2. 운영 데이터 로드와 feature 함수

In [3]:
def load_operational(substation_id: int) -> pd.DataFrame:
    path = f"{SOURCE_PREFIX}/operational_data/substation_{int(substation_id)}.csv"
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(path) as file:
            df = pd.read_csv(file, sep=";")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    missing_columns = [column for column in COMMON_SENSOR_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"missing common sensor columns for substation {substation_id}: {missing_columns}")
    return df.sort_values("timestamp").reset_index(drop=True)


OPERATIONAL_CACHE: dict[int, pd.DataFrame] = {}


def get_operational(substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in OPERATIONAL_CACHE:
        OPERATIONAL_CACHE[substation_id] = load_operational(substation_id)
    return OPERATIONAL_CACHE[substation_id]


def window_slice(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> pd.DataFrame:
    df = get_operational(substation_id)
    mask = df["timestamp"].ge(window_start) & df["timestamp"].lt(window_end)
    return df.loc[mask, ["timestamp", *COMMON_SENSOR_COLUMNS]].copy()


def expected_rows(window_start: pd.Timestamp, window_end: pd.Timestamp) -> int:
    minutes = (window_end - window_start).total_seconds() / 60
    return int(round(minutes / 10))


def disturbance_summary(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> tuple[int, str]:
    mask = (
        disturbances["substation ID"].eq(int(substation_id))
        & disturbances["Event start"].ge(window_start)
        & disturbances["Event start"].lt(window_end)
    )
    window_disturbances = disturbances.loc[mask].copy()
    if window_disturbances.empty:
        return 0, ""
    disturbance_types = "|".join(sorted(window_disturbances["type"].dropna().astype(str).unique()))
    return int(len(window_disturbances)), disturbance_types


def feature_values(window_data: pd.DataFrame) -> dict[str, float]:
    values: dict[str, float] = {}
    for sensor in COMMON_SENSOR_COLUMNS:
        series = pd.to_numeric(window_data[sensor], errors="coerce")
        non_null = series.dropna()
        values[f"{sensor}__mean"] = float(non_null.mean()) if len(non_null) else np.nan
        values[f"{sensor}__std"] = float(non_null.std(ddof=1)) if len(non_null) > 1 else 0.0 if len(non_null) == 1 else np.nan
        values[f"{sensor}__min"] = float(non_null.min()) if len(non_null) else np.nan
        values[f"{sensor}__max"] = float(non_null.max()) if len(non_null) else np.nan
        values[f"{sensor}__median"] = float(non_null.median()) if len(non_null) else np.nan
        values[f"{sensor}__last_minus_first"] = float(non_null.iloc[-1] - non_null.iloc[0]) if len(non_null) > 1 else 0.0 if len(non_null) == 1 else np.nan
        values[f"{sensor}__missing_rate"] = float(series.isna().mean()) if len(series) else 1.0
    return values


def make_sample(
    *,
    label: str,
    source_event_id: int,
    substation_id: int,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    metadata: dict | None = None,
) -> dict:
    metadata = metadata or {}
    window_data = window_slice(substation_id, window_start, window_end)
    expected_sample_count = expected_rows(window_start, window_end)
    sample_count = int(len(window_data))
    coverage_rate = float(sample_count / expected_sample_count) if expected_sample_count else 0.0
    disturbance_count, disturbance_types = disturbance_summary(substation_id, window_start, window_end)

    row = {
        "sample_id": f"{label}_{int(source_event_id)}",
        "label": label,
        "source_event_id": int(source_event_id),
        "substation_id": int(substation_id),
        "window_start": window_start,
        "window_end": window_end,
        "window_days": float((window_end - window_start).total_seconds() / 86400),
        "sample_count": sample_count,
        "expected_sample_count": expected_sample_count,
        "coverage_rate": coverage_rate,
        "disturbance_count": disturbance_count,
        "disturbance_types": disturbance_types,
    }
    row.update(metadata)
    row.update(feature_values(window_data))
    return row

## 3. Audit와 feature dataset 생성

In [4]:
normal_rows = []
for _, event in normal_events.sort_values("Event ID").iterrows():
    normal_rows.append(
        make_sample(
            label="normal",
            source_event_id=int(event["Event ID"]),
            substation_id=int(event["substation ID"]),
            window_start=event["Event start"],
            window_end=event["Event end"],
            metadata={
                "source_type": "normal_event",
                "fault_label": "",
                "fault_label_known": False,
                "fault_label_unknown": False,
                "efd_possible": False,
                "training_metadata_complete": pd.notna(event["Training start"]) and pd.notna(event["Training end"]),
                "event67_flag": False,
                "audit_status": "normal_reference",
                "train_status": "accepted",
            },
        )
    )

fault_rows = []
for _, event in faults.sort_values("Event ID").iterrows():
    event_id = int(event["Event ID"])
    substation_id = int(event["substation ID"])
    report_date = event["Report date"]
    window_start = report_date - pd.Timedelta(days=7)
    window_end = report_date
    fault_label = "" if pd.isna(event["Fault label"]) else str(event["Fault label"])
    fault_label_unknown = fault_label.strip().lower() == "unknown"
    training_metadata_complete = pd.notna(event["Training start"]) and pd.notna(event["Training end"])

    if event_id == EVENT20_ID:
        audit_status = "event20_low_coverage_excluded"
        train_status = "excluded"
    elif event_id == EVENT69_ID:
        audit_status = "event69_training_end_excluded"
        train_status = "excluded"
    else:
        audit_status = "trainable_fault_event"
        train_status = "accepted"

    anomaly_start = event["Possible anomaly start"]
    anomaly_end = event["Possible anomaly end"]
    anomaly_start_to_report_days = (
        float((report_date - anomaly_start).total_seconds() / 86400) if pd.notna(anomaly_start) else np.nan
    )
    anomaly_duration_days = (
        float((anomaly_end - anomaly_start).total_seconds() / 86400)
        if pd.notna(anomaly_start) and pd.notna(anomaly_end)
        else np.nan
    )
    training_gap_to_report_days = (
        float((report_date - event["Training end"]).total_seconds() / 86400)
        if pd.notna(event["Training end"])
        else np.nan
    )

    fault_rows.append(
        make_sample(
            label=POSITIVE_LABEL,
            source_event_id=event_id,
            substation_id=substation_id,
            window_start=window_start,
            window_end=window_end,
            metadata={
                "source_type": "fault_record",
                "report_date": report_date,
                "possible_anomaly_start": anomaly_start,
                "possible_anomaly_end": anomaly_end,
                "training_start": event["Training start"],
                "training_end": event["Training end"],
                "fault_label": fault_label,
                "fault_label_known": not fault_label_unknown,
                "fault_label_unknown": fault_label_unknown,
                "efd_possible": bool(event["efd_possible"]),
                "training_metadata_complete": bool(training_metadata_complete),
                "event67_flag": event_id == EVENT67_ID,
                "audit_status": audit_status,
                "train_status": train_status,
                "anomaly_start_to_report_days": anomaly_start_to_report_days,
                "anomaly_duration_days": anomaly_duration_days,
                "training_gap_to_report_days": training_gap_to_report_days,
            },
        )
    )

normal_df = pd.DataFrame(normal_rows)
fault_audit_df = pd.DataFrame(fault_rows)
audit_df = fault_audit_df.copy()
feature_pool_df = pd.concat(
    [
        pd.DataFrame(normal_rows),
        fault_audit_df.loc[fault_audit_df["train_status"].eq("accepted")],
    ],
    ignore_index=True,
)

audit_df.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")
feature_pool_df.to_csv(FEATURES_PATH, index=False, encoding="utf-8-sig")

print("audit rows:", len(audit_df))
print("feature pool rows:", len(feature_pool_df))
print(audit_df[["source_event_id", "substation_id", "fault_label", "fault_label_unknown", "training_metadata_complete", "event67_flag", "coverage_rate", "audit_status"]].to_string(index=False))

audit rows: 33
feature pool rows: 66
 source_event_id  substation_id                                                            fault_label  fault_label_unknown  training_metadata_complete  event67_flag  coverage_rate                  audit_status
               1             10             Motorised control valve (primary side): Actuator defective                False                        True         False       1.000000         trainable_fault_event
               3             12                               Control unit: Incorrect parameterisation                False                        True         False       1.000000         trainable_fault_event
               5             11                                    Failure of the heating circuit pump                False                        True         False       1.000000         trainable_fault_event
               6             21                               Control unit: Incorrect parameterisation                F

## 4. 학습 dataset 정의

In [5]:
def build_dataset(dataset_id: str) -> pd.DataFrame:
    normal_part = feature_pool_df.loc[feature_pool_df["label"].eq("normal")].copy()
    fault_part = feature_pool_df.loc[feature_pool_df["label"].eq(POSITIVE_LABEL)].copy()

    if dataset_id == "fault_event_known_only":
        fault_part = fault_part.loc[~fault_part["source_event_id"].isin([EVENT34_ID])]
    elif dataset_id == "fault_event_with_unknown34":
        pass
    elif dataset_id == "fault_event_no_event67":
        fault_part = fault_part.loc[~fault_part["source_event_id"].isin([EVENT67_ID])]
    else:
        raise ValueError(dataset_id)

    dataset = pd.concat([normal_part, fault_part], ignore_index=True)
    dataset["dataset_id"] = dataset_id
    return dataset


DATASETS = {
    dataset_id: build_dataset(dataset_id)
    for dataset_id in [
        "fault_event_known_only",
        "fault_event_with_unknown34",
        "fault_event_no_event67",
    ]
}


summary_rows = []
for dataset_id, dataset in DATASETS.items():
    positive_events = sorted(
        dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int).tolist()
    )
    summary_rows.append(
        {
            "dataset_id": dataset_id,
            "rows": int(len(dataset)),
            "normal_rows": int(dataset["label"].eq("normal").sum()),
            "positive_rows": int(dataset["label"].eq(POSITIVE_LABEL).sum()),
            "positive_event_ids": "|".join(map(str, positive_events)),
            "contains_event20": EVENT20_ID in positive_events,
            "contains_event34": EVENT34_ID in positive_events,
            "contains_event67": EVENT67_ID in positive_events,
            "contains_event69": EVENT69_ID in positive_events,
            "unknown_fault_rows": int(
                dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "fault_label_unknown"].sum()
            ),
            "incomplete_training_metadata_rows": int(
                dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "training_metadata_complete"].eq(False).sum()
            ),
            "min_coverage_rate": float(dataset["coverage_rate"].min()),
            "median_coverage_rate": float(dataset["coverage_rate"].median()),
        }
    )

dataset_summary_df = pd.DataFrame(summary_rows)
dataset_summary_df.to_csv(DATASET_SUMMARY_PATH, index=False, encoding="utf-8-sig")
dataset_summary_df

,dataset_id,rows,normal_rows,positive_rows,positive_event_ids,contains_event20,contains_event34,contains_event67,contains_event69,unknown_fault_rows,incomplete_training_metadata_rows,min_coverage_rate,median_coverage_rate
0,fault_event_known_only,65,35,30,1|3|5|6|7|10|11|13|15|23|24|29|32|36|37|38|40|...,False,False,True,False,0,4,0.99504,1.0
1,fault_event_with_unknown34,66,35,31,1|3|5|6|7|10|11|13|15|23|24|29|32|34|36|37|38|...,False,True,True,False,1,4,0.99504,1.0
2,fault_event_no_event67,65,35,30,1|3|5|6|7|10|11|13|15|23|24|29|32|34|36|37|38|...,False,True,False,False,1,4,0.99504,1.0


## 5. Group CV 학습

In [6]:
def make_models() -> dict[str, Pipeline]:
    return {
        "dummy_most_frequent": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "logistic_balanced": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def positive_scores(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        classes = list(estimator.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
    return np.zeros(len(X), dtype=float)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if y_score is not None and len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_score))
    else:
        roc_auc = 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_and_evaluate(dataset_id: str, dataset: pd.DataFrame):
    dataset = dataset.copy().reset_index(drop=True)
    X = dataset[FEATURE_COLUMNS].copy()
    y = dataset["label"].map(LABEL_TO_BINARY).astype(int).to_numpy()
    groups = dataset["substation_id"].astype(str).to_numpy()

    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    metric_rows = []
    prediction_rows = []
    importance_rows = []

    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]
        train_groups = sorted(set(groups[train_index]))
        test_groups = sorted(set(groups[test_index]))
        assert set(train_groups).isdisjoint(test_groups)
        assert len(np.unique(y_train)) == 2

        for model_name, model in make_models().items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test).astype(int)
            y_score = positive_scores(model, X_test)
            row = metric_row(y_test, y_pred, y_score)
            row.update(
                {
                    "dataset_id": dataset_id,
                    "model": model_name,
                    "fold": fold,
                    "train_rows": int(len(train_index)),
                    "test_rows": int(len(test_index)),
                    "train_groups": "|".join(train_groups),
                    "test_groups": "|".join(test_groups),
                    "test_normal": int((y_test == 0).sum()),
                    "test_fault_event": int((y_test == 1).sum()),
                }
            )
            metric_rows.append(row)

            prediction_meta = dataset.iloc[test_index][
                [
                    "sample_id",
                    "label",
                    "source_event_id",
                    "substation_id",
                    "window_start",
                    "window_end",
                    "window_days",
                    "coverage_rate",
                    "sample_count",
                    "fault_label",
                    "fault_label_known",
                    "fault_label_unknown",
                    "efd_possible",
                    "training_metadata_complete",
                    "event67_flag",
                    "disturbance_count",
                ]
            ].copy()
            prediction_meta["dataset_id"] = dataset_id
            prediction_meta["fold"] = fold
            prediction_meta["model"] = model_name
            prediction_meta["actual_binary"] = y_test
            prediction_meta["predicted_binary"] = y_pred
            prediction_meta["predicted_label"] = [BINARY_TO_LABEL[int(value)] for value in y_pred]
            prediction_meta["positive_score"] = y_score
            prediction_meta["is_correct"] = prediction_meta["actual_binary"].eq(
                prediction_meta["predicted_binary"]
            )
            prediction_rows.extend(prediction_meta.to_dict("records"))

            if model_name == "logistic_balanced":
                coefficients = model.named_steps["model"].coef_[0]
                for feature, coefficient in zip(FEATURE_COLUMNS, coefficients):
                    importance_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "fold": fold,
                            "feature": feature,
                            "coefficient": float(coefficient),
                            "abs_coefficient": float(abs(coefficient)),
                        }
                    )

    return metric_rows, prediction_rows, importance_rows


all_metric_rows = []
all_prediction_rows = []
all_importance_rows = []
for dataset_id, dataset in DATASETS.items():
    metric_rows, prediction_rows, importance_rows = train_and_evaluate(dataset_id, dataset)
    all_metric_rows.extend(metric_rows)
    all_prediction_rows.extend(prediction_rows)
    all_importance_rows.extend(importance_rows)

cv_metrics_df = pd.DataFrame(all_metric_rows)
cv_predictions_df = pd.DataFrame(all_prediction_rows)
feature_importance_raw_df = pd.DataFrame(all_importance_rows)

cv_metrics_df.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")
cv_predictions_df.to_csv(CV_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

feature_importance_df = (
    feature_importance_raw_df.groupby(["dataset_id", "feature"], as_index=False)
    .agg(
        mean_coefficient=("coefficient", "mean"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        std_coefficient=("coefficient", "std"),
    )
    .sort_values(["dataset_id", "mean_abs_coefficient"], ascending=[True, False])
)
feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")

cv_metrics_df.groupby(["dataset_id", "model"])[
    ["balanced_accuracy", "precision", "recall", "f1", "roc_auc", "fp", "fn"]
].mean().round(4)

balanced_accuracy  precision  \
dataset_id                 model                                               
fault_event_known_only     dummy_most_frequent             0.5000     0.0000   
                           logistic_balanced               0.5363     0.5022   
fault_event_no_event67     dummy_most_frequent             0.5000     0.0000   
                           logistic_balanced               0.4304     0.3581   
fault_event_with_unknown34 dummy_most_frequent             0.5000     0.0000   
                           logistic_balanced               0.4167     0.3343   

                                                recall      f1  roc_auc   fp  \
dataset_id                 model                                               
fault_event_known_only     dummy_most_frequent  0.0000  0.0000   0.5000  0.0   
                           logistic_balanced    0.5171  0.4824   0.5305  3.2   
fault_event_no_event67     dummy_most_frequent  0.0000  0.0000   0.5000  0.0   
                           logistic_balanced    0.4000  0.3737   0.3966  3.8   
fault_event_with_unknown34 dummy_most_frequent  0.0000  0.0000   0.5000  0.0   
                           logistic_balanced    0.3190  0.3231   0.3748  3.4   

                                                 fn  
dataset_id                 model                     
fault_event_known_only     dummy_most_frequent  6.0  
                           logistic_balanced    3.0  
fault_event_no_event67     dummy_most_frequent  6.0  
                           logistic_balanced    3.6  
fault_event_with_unknown34 dummy_most_frequent  6.2  
                           logistic_balanced    4.2

## 6. Threshold별 tradeoff

In [7]:
threshold_rows = []
for dataset_id in DATASETS:
    prediction_part = cv_predictions_df.loc[
        (cv_predictions_df["dataset_id"].eq(dataset_id))
        & (cv_predictions_df["model"].eq("logistic_balanced"))
    ].copy()
    y_true = prediction_part["actual_binary"].astype(int).to_numpy()
    y_score = prediction_part["positive_score"].astype(float).to_numpy()
    for threshold in THRESHOLDS:
        y_pred = (y_score >= threshold).astype(int)
        row = metric_row(y_true, y_pred, y_score)
        row.update(
            {
                "dataset_id": dataset_id,
                "model": "logistic_balanced",
                "threshold": threshold,
                "rows": int(len(prediction_part)),
                "normal_rows": int((y_true == 0).sum()),
                "positive_rows": int((y_true == 1).sum()),
                "false_positive_rate": float(row["fp"] / (row["fp"] + row["tn"]))
                if (row["fp"] + row["tn"])
                else 0.0,
                "false_negative_rate": float(row["fn"] / (row["fn"] + row["tp"]))
                if (row["fn"] + row["tp"])
                else 0.0,
            }
        )
        threshold_rows.append(row)

threshold_metrics_df = pd.DataFrame(threshold_rows)
threshold_metrics_df.to_csv(THRESHOLD_METRICS_PATH, index=False, encoding="utf-8-sig")
threshold_metrics_df.sort_values(["dataset_id", "threshold"]).round(4)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,dataset_id,model,threshold,rows,normal_rows,positive_rows,false_positive_rate,false_negative_rate
0,0.5077,0.5214,0.4773,0.7000,0.5676,0.5343,12,23,9,21,fault_event_known_only,logistic_balanced,0.3,65,35,30,0.6571,0.3000
1,0.5231,0.5262,0.4857,0.5667,0.5231,0.5343,17,18,13,17,fault_event_known_only,logistic_balanced,0.4,65,35,30,0.5143,0.4333
2,0.5231,0.5214,0.4839,0.5000,0.4918,0.5343,19,16,15,15,fault_event_known_only,logistic_balanced,0.5,65,35,30,0.4571,0.5000
3,0.5231,0.5167,0.4815,0.4333,0.4561,0.5343,21,14,17,13,fault_event_known_only,logistic_balanced,0.6,65,35,30,0.4000,0.5667
8,0.4000,0.4095,0.3902,0.5333,0.4507,0.3829,10,25,14,16,fault_event_no_event67,logistic_balanced,0.3,65,35,30,0.7143,0.4667
9,0.4000,0.4048,0.3784,0.4667,0.4179,0.3829,12,23,16,14,fault_event_no_event67,logistic_balanced,0.4,65,35,30,0.6571,0.5333
10,0.4308,0.4286,0.3871,0.4000,0.3934,0.3829,16,19,18,12,fault_event_no_event67,logistic_balanced,0.5,65,35,30,0.5429,0.6000
11,0.3846,0.3762,0.3077,0.2667,0.2857,0.3829,17,18,22,8,fault_event_no_event67,logistic_balanced,0.6,65,35,30,0.5143,0.7333
4,0.3939,0.3972,0.3784,0.4516,0.4118,0.3889,12,23,17,14,fault_event_with_unknown34,logistic_balanced,0.3,66,35,31,0.6571,0.5484
5,0.3788,0.3774,0.3438,0.3548,0.3492,0.3889,14,21,20,11,fault_event_with_unknown34,logistic_balanced,0.4,66,35,31,0.6000,0.6452


## 7. 의사결정 matrix와 보고서

In [8]:
def mean_metric(dataset_id: str, model: str, metric: str) -> float:
    values = cv_metrics_df.loc[
        cv_metrics_df["dataset_id"].eq(dataset_id) & cv_metrics_df["model"].eq(model), metric
    ]
    return float(values.mean()) if len(values) else np.nan


def threshold_metric(dataset_id: str, threshold: float, metric: str) -> float:
    values = threshold_metrics_df.loc[
        threshold_metrics_df["dataset_id"].eq(dataset_id)
        & threshold_metrics_df["threshold"].eq(threshold),
        metric,
    ]
    return float(values.iloc[0]) if len(values) else np.nan


decision_rows = []
for dataset_id in DATASETS:
    dataset = DATASETS[dataset_id]
    summary = dataset_summary_df.loc[dataset_summary_df["dataset_id"].eq(dataset_id)].iloc[0].to_dict()
    logistic_bacc = mean_metric(dataset_id, "logistic_balanced", "balanced_accuracy")
    dummy_bacc = mean_metric(dataset_id, "dummy_most_frequent", "balanced_accuracy")
    decision_rows.append(
        {
            "dataset_id": dataset_id,
            "rows": int(summary["rows"]),
            "normal_rows": int(summary["normal_rows"]),
            "positive_rows": int(summary["positive_rows"]),
            "contains_event34": bool(summary["contains_event34"]),
            "contains_event67": bool(summary["contains_event67"]),
            "dummy_balanced_accuracy_mean": dummy_bacc,
            "logistic_balanced_accuracy_mean": logistic_bacc,
            "balanced_accuracy_lift_vs_dummy": logistic_bacc - dummy_bacc,
            "logistic_precision_mean": mean_metric(dataset_id, "logistic_balanced", "precision"),
            "logistic_recall_mean": mean_metric(dataset_id, "logistic_balanced", "recall"),
            "logistic_f1_mean": mean_metric(dataset_id, "logistic_balanced", "f1"),
            "logistic_roc_auc_mean": mean_metric(dataset_id, "logistic_balanced", "roc_auc"),
            "threshold05_false_positive_rate": threshold_metric(dataset_id, 0.5, "false_positive_rate"),
            "threshold05_false_negative_rate": threshold_metric(dataset_id, 0.5, "false_negative_rate"),
            "unknown_fault_rows": int(summary["unknown_fault_rows"]),
            "incomplete_training_metadata_rows": int(summary["incomplete_training_metadata_rows"]),
        }
    )

decision_matrix_df = pd.DataFrame(decision_rows)
decision_matrix_df.to_csv(DECISION_MATRIX_PATH, index=False, encoding="utf-8-sig")

known = decision_matrix_df.loc[decision_matrix_df["dataset_id"].eq("fault_event_known_only")].iloc[0]
with_unknown = decision_matrix_df.loc[decision_matrix_df["dataset_id"].eq("fault_event_with_unknown34")].iloc[0]
without67 = decision_matrix_df.loc[decision_matrix_df["dataset_id"].eq("fault_event_no_event67")].iloc[0]

event34_bacc_delta = float(with_unknown["logistic_balanced_accuracy_mean"] - known["logistic_balanced_accuracy_mean"])
event34_fpr_delta = float(with_unknown["threshold05_false_positive_rate"] - known["threshold05_false_positive_rate"])
event67_bacc_delta = float(without67["logistic_balanced_accuracy_mean"] - with_unknown["logistic_balanced_accuracy_mean"])

event34_decision = (
    "Event 34 포함 허용 후보"
    if event34_bacc_delta >= -0.02 and event34_fpr_delta <= 0.05
    else "Event 34 audit 전용 유지 권장"
)

best_row = decision_matrix_df.sort_values(
    ["logistic_balanced_accuracy_mean", "logistic_f1_mean"], ascending=False
).iloc[0]

previous_path = OUTPUT_DIR / "m1_positive_expansion_threshold_metrics.csv"
previous_summary_text = "이전 확장 positive 실험 파일을 읽지 못해 직접 비교는 생략했다."
if previous_path.exists():
    previous_threshold = pd.read_csv(previous_path)
    if {"dataset_id", "threshold", "balanced_accuracy", "f1", "false_positive_rate"}.issubset(previous_threshold.columns):
        previous_best = previous_threshold.loc[
            previous_threshold["dataset_id"].eq("expanded_weak_positive")
            & previous_threshold["threshold"].eq(0.5)
        ]
        if len(previous_best):
            prev = previous_best.iloc[0]
            current = threshold_metrics_df.loc[
                threshold_metrics_df["dataset_id"].eq(str(best_row["dataset_id"]))
                & threshold_metrics_df["threshold"].eq(0.5)
            ].iloc[0]
            previous_summary_text = (
                f"기존 expanded weak positive의 threshold 0.5 balanced accuracy는 {prev['balanced_accuracy']:.4f}, "
                f"f1은 {prev['f1']:.4f}, false positive rate는 {prev['false_positive_rate']:.4f}였다. "
                f"이번 best dataset({best_row['dataset_id']})의 threshold 0.5 balanced accuracy는 {current['balanced_accuracy']:.4f}, "
                f"f1은 {current['f1']:.4f}, false positive rate는 {current['false_positive_rate']:.4f}이다."
            )


def metric_table(df: pd.DataFrame, float_cols: list[str]) -> str:
    table = df.copy()
    for column in float_cols:
        if column in table.columns:
            table[column] = table[column].map(lambda value: f"{value:.4f}")
    columns = list(table.columns)
    lines = [
        "| " + " | ".join(str(column) for column in columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for _, row in table.iterrows():
        rendered_values = []
        for column in columns:
            value = row[column]
            if pd.isna(value):
                rendered_values.append("")
            else:
                rendered_values.append(str(value).replace("|", "\\|"))
        lines.append("| " + " | ".join(rendered_values) + " |")
    return "\n".join(lines)


cv_summary = (
    cv_metrics_df.groupby(["dataset_id", "model"], as_index=False)
    .agg(
        balanced_accuracy=("balanced_accuracy", "mean"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        roc_auc=("roc_auc", "mean"),
        fp=("fp", "mean"),
        fn=("fn", "mean"),
    )
    .sort_values(["dataset_id", "model"])
)

top_features = (
    feature_importance_df.sort_values(["dataset_id", "mean_abs_coefficient"], ascending=[True, False])
    .groupby("dataset_id")
    .head(8)
    .reset_index(drop=True)
)

incomplete_fault_ids = sorted(
    audit_df.loc[~audit_df["training_metadata_complete"], "source_event_id"].astype(int).tolist()
)

report = f'''
# M1 normal vs fault event 보고서

## 결론

- 이번 실험은 조기탐지가 아니라 `normal`과 `fault_event` 상태를 구분하는 재라벨링 실험이다.
- `fault_event`는 M1 `faults.csv`에 존재하는 fault record로 정의했다.
- `unknown` fault label은 negative나 비라벨 이벤트로 단정하지 않았다. Event 34는 `fault_label_unknown=True`인 fault event로 audit에 기록했고, 포함/제외 민감도로만 판단했다.
- Event 20은 7일 window coverage가 낮아 학습에서 제외하고 audit에만 남겼다.
- Event 69는 요청 기준에 따라 학습에서 제외하고 audit에만 남겼다.
- 현재 최고 평균 balanced accuracy dataset은 `{best_row['dataset_id']}`이며, logistic 평균 balanced accuracy는 {best_row['logistic_balanced_accuracy_mean']:.4f}, 평균 f1은 {best_row['logistic_f1_mean']:.4f}이다.
- Event 34 판단: {event34_decision}. 포함 시 balanced accuracy 변화는 {event34_bacc_delta:+.4f}, threshold 0.5 false positive rate 변화는 {event34_fpr_delta:+.4f}이다.
- Event 67 제외 민감도: `fault_event_with_unknown34` 대비 balanced accuracy 변화는 {event67_bacc_delta:+.4f}이다.
- 최종 판단은 목표 전환 보류다. `fault_event_known_only`만 Dummy보다 소폭 개선됐고, Event 34 포함 dataset과 Event 67 제외 dataset은 모두 불안정했다. 기존 expanded weak positive 기준보다 threshold 0.5 성능도 좋아지지 않아 다음 단계는 모델 변경보다 feature 확장 또는 normal negative 재설계가 우선이다.

## 데이터 구성

| 항목 | 값 |
|---|---:|
| M1 fault audit row | {len(audit_df)} |
| normal reference row | {len(normal_df)} |
| feature pool row | {len(feature_pool_df)} |
| 학습 feature 수 | {len(FEATURE_COLUMNS)} |
| 센서 수 | {len(COMMON_SENSOR_COLUMNS)} |
| 통계 수 | {len(SENSOR_STATS)} |

## Dataset Summary

{metric_table(dataset_summary_df, ['min_coverage_rate', 'median_coverage_rate'])}

## Group CV 평균 성능

{metric_table(cv_summary, ['balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'fp', 'fn'])}

## Threshold 0.3~0.6 성능

{metric_table(threshold_metrics_df.sort_values(['dataset_id', 'threshold']), ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'false_positive_rate', 'false_negative_rate'])}

## Decision Matrix

{metric_table(decision_matrix_df, ['dummy_balanced_accuracy_mean', 'logistic_balanced_accuracy_mean', 'balanced_accuracy_lift_vs_dummy', 'logistic_precision_mean', 'logistic_recall_mean', 'logistic_f1_mean', 'logistic_roc_auc_mean', 'threshold05_false_positive_rate', 'threshold05_false_negative_rate'])}

## 이전 확장 positive 실험과의 참고 비교

{previous_summary_text}

## 주요 feature 후보

{metric_table(top_features, ['mean_coefficient', 'mean_abs_coefficient', 'std_coefficient'])}

## Audit 해석 메모

- Event 34는 `fault_label_unknown=True`이고 7일 window coverage가 {float(audit_df.loc[audit_df['source_event_id'].eq(EVENT34_ID), 'coverage_rate'].iloc[0]):.4f}이다.
- Event 67은 장기 anomaly flag로만 기록했고, 학습 metadata로 모델 입력에는 넣지 않았다.
- Training metadata 불완전 fault event ID는 {incomplete_fault_ids}이다. 이번 실험에서는 요청된 Event 69만 학습 제외로 고정하고, 나머지는 `training_metadata_complete=False`로 audit에 남겼다.
- fault metadata, 날짜, event id, `substation_id`, coverage는 학습 입력에서 제외했다.

## 생성 산출물

- `06_노트북/07_m1_normal_vs_fault_event_training.ipynb`
- `07_데이터산출물/m1_fault_event_audit.csv`
- `07_데이터산출물/m1_fault_event_features.csv`
- `07_데이터산출물/m1_fault_event_dataset_summary.csv`
- `07_데이터산출물/m1_fault_event_cv_metrics.csv`
- `07_데이터산출물/m1_fault_event_cv_predictions.csv`
- `07_데이터산출물/m1_fault_event_threshold_metrics.csv`
- `07_데이터산출물/m1_fault_event_feature_importance.csv`
- `07_데이터산출물/m1_fault_event_decision_matrix.csv`

## 검증 결과

- M1 `faults.csv` 전체 33건을 audit에 기록했다.
- Event 20과 Event 69는 학습 dataset에서 제외되고 audit에만 남았다.
- Event 34는 audit에 `fault_label_unknown=True`로 기록되며, `fault_event_known_only`에서는 제외되고 `fault_event_with_unknown34`에는 포함된다.
- `fault_event_known_only`: normal 35 + positive 30
- `fault_event_with_unknown34`: normal 35 + positive 31
- `fault_event_no_event67`: normal 35 + positive 30
- group CV에서 같은 `substation_id`가 train/test에 동시에 들어가지 않음을 fold별 assert로 확인했다.
- 학습 입력은 M1 공통 10개 센서 기반 70개 feature만 사용했다.
'''.strip()

REPORT_PATH.write_text(report, encoding="utf-8")
decision_matrix_df

,dataset_id,rows,normal_rows,positive_rows,contains_event34,contains_event67,dummy_balanced_accuracy_mean,logistic_balanced_accuracy_mean,balanced_accuracy_lift_vs_dummy,logistic_precision_mean,logistic_recall_mean,logistic_f1_mean,logistic_roc_auc_mean,threshold05_false_positive_rate,threshold05_false_negative_rate,unknown_fault_rows,incomplete_training_metadata_rows
0,fault_event_known_only,65,35,30,False,True,0.5,0.536349,0.036349,0.502222,0.517143,0.482398,0.530476,0.457143,0.500000,0,4
1,fault_event_with_unknown34,66,35,31,True,True,0.5,0.416667,-0.083333,0.334286,0.319048,0.323077,0.374830,0.485714,0.677419,1,4
2,fault_event_no_event67,65,35,30,True,False,0.5,0.430357,-0.069643,0.358095,0.400000,0.373706,0.396627,0.542857,0.600000,1,4


## 8. 최종 검증

In [9]:
assert len(audit_df) == 33
assert len(normal_df) == 35
assert len(FEATURE_COLUMNS) == 70
assert set(feature for feature in FEATURE_COLUMNS) == {
    f"{sensor}__{stat}" for sensor in COMMON_SENSOR_COLUMNS for stat in SENSOR_STATS
}

training_fault_ids = set(
    feature_pool_df.loc[feature_pool_df["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int)
)
assert EVENT20_ID not in training_fault_ids
assert EVENT69_ID not in training_fault_ids
assert EVENT34_ID in training_fault_ids

assert audit_df["source_event_id"].astype(int).nunique() == 33
assert audit_df.loc[audit_df["source_event_id"].eq(EVENT20_ID), "audit_status"].iloc[0] == "event20_low_coverage_excluded"
assert audit_df.loc[audit_df["source_event_id"].eq(EVENT69_ID), "audit_status"].iloc[0] == "event69_training_end_excluded"
assert bool(audit_df.loc[audit_df["source_event_id"].eq(EVENT34_ID), "fault_label_unknown"].iloc[0]) is True

known_dataset = DATASETS["fault_event_known_only"]
with_unknown_dataset = DATASETS["fault_event_with_unknown34"]
no_event67_dataset = DATASETS["fault_event_no_event67"]

assert int(known_dataset["label"].eq("normal").sum()) == 35
assert int(known_dataset["label"].eq(POSITIVE_LABEL).sum()) == 30
assert EVENT34_ID not in set(known_dataset.loc[known_dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))

assert int(with_unknown_dataset["label"].eq("normal").sum()) == 35
assert int(with_unknown_dataset["label"].eq(POSITIVE_LABEL).sum()) == 31
assert EVENT34_ID in set(with_unknown_dataset.loc[with_unknown_dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))

assert int(no_event67_dataset["label"].eq("normal").sum()) == 35
assert int(no_event67_dataset["label"].eq(POSITIVE_LABEL).sum()) == 30
assert EVENT67_ID not in set(no_event67_dataset.loc[no_event67_dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))

assert len(cv_metrics_df) == 3 * 5 * 2
assert len(threshold_metrics_df) == 3 * len(THRESHOLDS)
assert len(feature_importance_df) == 3 * len(FEATURE_COLUMNS)

for _, row in cv_metrics_df.iterrows():
    train_groups = set(str(row["train_groups"]).split("|"))
    test_groups = set(str(row["test_groups"]).split("|"))
    assert train_groups.isdisjoint(test_groups)

metadata_like = {
    "sample_id",
    "label",
    "source_event_id",
    "substation_id",
    "window_start",
    "window_end",
    "window_days",
    "sample_count",
    "expected_sample_count",
    "coverage_rate",
    "disturbance_count",
    "disturbance_types",
    "source_type",
    "report_date",
    "possible_anomaly_start",
    "possible_anomaly_end",
    "training_start",
    "training_end",
    "fault_label",
    "fault_label_known",
    "fault_label_unknown",
    "efd_possible",
    "training_metadata_complete",
    "event67_flag",
    "audit_status",
    "train_status",
    "anomaly_start_to_report_days",
    "anomaly_duration_days",
    "training_gap_to_report_days",
    "dataset_id",
}
feature_like_columns = [
    column
    for column in feature_pool_df.columns
    if column not in metadata_like
]
assert sorted(feature_like_columns) == sorted(FEATURE_COLUMNS)

print("validation passed")
print("outputs:")
for path in [
    AUDIT_PATH,
    FEATURES_PATH,
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
    REPORT_PATH,
]:
    print(path.relative_to(ROOT))

validation passed
outputs:
07_데이터산출물\m1_fault_event_audit.csv
07_데이터산출물\m1_fault_event_features.csv
07_데이터산출물\m1_fault_event_dataset_summary.csv
07_데이터산출물\m1_fault_event_cv_metrics.csv
07_데이터산출물\m1_fault_event_cv_predictions.csv
07_데이터산출물\m1_fault_event_threshold_metrics.csv
07_데이터산출물\m1_fault_event_feature_importance.csv
07_데이터산출물\m1_fault_event_decision_matrix.csv
07_데이터산출물\07_M1_normal_vs_fault_event_보고서.md
